# Process stock fasta sequence
Does the stock sequence fasta have indels relative to the reference sequence? Indels can cause issues for variant calling because the gene annotation file relies on process gene coordinates which can be resolved by removing indels relative to the reference sequence

In [1]:
import os
import pandas as pd
from Bio import Align
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

Notebook parameters:

In [2]:
if 'snakemake' in globals():
    virus=snakemake.wildcards.virus
    sample=snakemake.wildcards.sample
    input_stock = snakemake.input.input_stock
    input_reference = snakemake.input.input_reference
    output_stock = snakemake.output.output_stock
else:
    virus="Foundry-HMPV"
    sample="Stock"
    input_stock = f"../../results/consensus/{virus}/{sample}.fa"
    input_reference = f"../../results/reference/{virus}/bwa/{virus}.fasta"
    os.makedirs(f"../../results/consensus_no_indels/{virus}/", exist_ok=True)
    output_stock = f"../../results/consensus_no_indels/{virus}/{sample}.fa"

Align stock and reference sequences

In [3]:
aligner = Align.PairwiseAligner()
# Alignment settings to match SnapGene defaults
aligner.mode = "local"
print(aligner.algorithm)
aligner.match_score = 1
aligner.mismatch_score = 0
aligner.open_gap_score = -10
aligner.extend_gap_score = -1

seq1 = SeqIO.read(input_reference, "fasta")
seq2 = SeqIO.read(input_stock, "fasta")

alignment = aligner.align(seq1, seq2)[0]
print(alignment)

Smith-Waterman
AY297749.         0 ACGCGAAAAAAACGCGTATAAATTAAGTTACAAAAAAACATGGGACAAGTGAAAATGTCT
                  0 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Consensus         0 ACGCGAAAAAAACGCGTATAAATTAAGTTACAAAAAAACATGGGACAAGTGAAAATGTCT

AY297749.        60 CTTCAAGGGATTCACCTGAGTGATCTATCATACAAGCATGCTATATTAAAAGAGTCTCAG
                 60 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Consensus        60 CTTCAAGGGATTCACCTGAGTGATCTATCATACAAGCATGCTATATTAAAAGAGTCTCAG

AY297749.       120 TATACAATAAAGAGAGATGTAGGCACAACAACAGCAGTGACACCCTCATCATTGCAACAA
                120 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Consensus       120 TATACAATAAAGAGAGATGTAGGCACAACAACAGCAGTGACACCCTCATCATTGCAACAA

AY297749.       180 GAAATAACACTATTGTGTGGAGAAATTCTATATGCTAAGCATGCTGATTACAAATATGCT
                180 ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Consensus       180 GAAATAACACTATTGTGTGGAGAAATTCTATATGCTAAGCATGCTGATTACAAATATGCT

AY297749.

Process alignment and remove gaps relative to reference sequence

In [4]:
aligned_ref = alignment[0]
aligned_stock = alignment[1]
new_stock_seq = ""
for index,char in enumerate(aligned_ref):
    if char == "-":
        print(f"Gap found at position {index+1}")
    else:
        new_stock_seq += aligned_stock[index]

# Write processed fasta sequence to file
record = SeqRecord(
    Seq(new_stock_seq), 
    id="consensus",
)

with open(output_stock, "w") as handle:
    SeqIO.write(record, handle, "fasta")

Gap found at position 2160
